# 01 — Drift lab

In this noteboook, we play with drifting iterations

$$q^{k+1}=(\mathrm{Id}+\tau V_{p,q^k})_\#q^k$$

on finite 2D clouds.  The datasets, fields, kernels are given by:

| group | options |
|---|---|
| dataset | `two_gaussians`, `ring`, `moons`, `checkerboard`, `imbalanced`, `thin_circle`, `mass_hierarchy`, `gaussian_aniso`|
| kernel | `Gaussian`, `Laplacian`, `Matern`, `IMQ`; `SIGMA`|
| field | `mean_shift`, `deng_official`, `sinkhorn_proxy`, `sharp_core`, `ms_unnorm`, `mmd`, `pgr`, `power`, `common_normalization`, `alpha`, `ot`, `linear`, `sinkhorn`, `flow_matching` |


## 0 · Setup and numerical self-check

The check table validates identities by finite differences, closed forms or
independent formulas.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
if os.path.basename(os.getcwd()) == "notebooks":   
    os.chdir("..")                                
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from msflow import *
import msflow

pd.set_option("display.width", 160)

In [ ]:
import msflow.checks as mathematical_checks

audit = checks_table(mathematical_checks.run_all())
print(f"{(audit['status'] == 'PASS').sum()} / {len(audit)} identities verified")
audit

---
## 1 · Config

``mean_shift`` is our implementation following the theoretical field.

``deng_official`` is the implementation used by Deng, Algorithm 2. Use ``KWARGS : "self_mask": True, "self_mask_distance": 1e6``


In [ ]:
# Data
DATASET = "ring"
DATASET_KWARGS = {}
N, SEED = 320, 2026

# Drift, Kernel, Bandwidth
DRIFT, KERNEL, SIGMA = "mean_shift", "Laplacian", 1.0
DRIFT_KWARGS = {}

# Euler
TAU, N_ITER, CLIP = 0.04, 600, None
PROBE_EVERY, STORE_EVERY = 20, 5

# Diagnostics and display
WITH_CERTIFICATES = True
EVAL_SIGMA, R_COVER = 0.5, 1.4
PLOT_KEYS = ("W2", "beta_star", "L_hat", "ot_alignment", "n_covered", "H")
PLOT_NCOL, PLOT_FIGSIZE = 3, (12, 6)
SNAPSHOT_KS = (0, N_ITER // 10, N_ITER // 3, N_ITER)
SNAPSHOT_PANEL_SIZE = (1.8, 1.7)
ANIMATION_GRID = 18
ANIMATION_EVERY_ITER, ANIMATION_INTERVAL_MS = 40, 70
ANIMATION_FIGSIZE, ANIMATION_MAX_WIDTH = (4.8, 4.8), 500
GALLERY_N, GALLERY_NCOL, GALLERY_PANEL_SIZE = min(N, 320), 4, 2.25
ZOO_PAGE_SIZE = 3
ZOO_WITH_CERTIFICATES = False 

# Bandwidth annealing
BANDWIDTH_DRIFT, BANDWIDTH_KERNEL = "mean_shift", KERNEL
BANDWIDTH_DRIFT_KWARGS = {}
BANDWIDTHS = (0.3, 0.7, 1.5, 3.0)
ANNEAL_START, ANNEAL_END, ANNEAL_KAPPA = 3.0, 0.3, N_ITER / 4

data = load_dataset(DATASET, n=N, seed=SEED, **DATASET_KWARGS)
field = make_drift(DRIFT, KERNEL, SIGMA, **DRIFT_KWARGS)
print(data.summary())
print(field)

fig = plot_dataset(data, figsize=(9, 6))


ax = plot_kernel_profiles(sigma=SIGMA)
save(ax[0].figure, "01_kernels");

---
## 2 . Running simulations

We record energy values $W_2^2(q^k,p)$ and $\mathrm{KL}(q^k_\sigma \,||\, p_\sigma)$ as well as convergence / regularity proxies:

$$\hat\alpha=-\mathfrak D/W_2^2,\qquad
  \hat L=\|V\|_{L^2(q)}/W_2,\qquad
  \beta_\star=-\mathfrak D/\|V\|^2,$$

In [ ]:
main_kernel = field_kernel(field)
main_probes = standard_probes(
    data, main_kernel, with_certificates=WITH_CERTIFICATES,
    eval_sigma=EVAL_SIGMA, r_cover=R_COVER, tau=TAU)

runs = Runs()
runs["main"] = run(
    data.q0, data.p, field, n_iter=N_ITER, tau=TAU, clip=CLIP,
    probes=main_probes, probe_every=PROBE_EVERY, store_every=STORE_EVERY,
    label=f"{DRIFT} / {KERNEL}")

fig = plot_runs(runs, keys=PLOT_KEYS, ncol=PLOT_NCOL, figsize=PLOT_FIGSIZE)
save(fig, "01_single_run")
runs_table(runs, n_modes=len(data.centers), tau=TAU)

In [ ]:
fig = plot_snapshots(runs, data.p, ks=SNAPSHOT_KS,
                     centers=data.centers, extent=data.extent,
                     r_cover=R_COVER, panel_size=SNAPSHOT_PANEL_SIZE)
save(fig, "01_snapshots");

### Animating the dynamics


In [ ]:
anim = animate_field(
    runs["main"], data.p, lambda k: field, extent=data.extent,
    n_grid=ANIMATION_GRID, every_iter=ANIMATION_EVERY_ITER,
    interval=ANIMATION_INTERVAL_MS, title=f"{DRIFT} / {KERNEL}",
    figsize=ANIMATION_FIGSIZE)
animation_html(anim, max_width=ANIMATION_MAX_WIDTH)

---
## 3 · Comparing fields


In [ ]:
laplace_kern = get_kernel("Laplacian", SIGMA)
proxy_kern = get_kernel("Gaussian", SIGMA)

deng_theoretical = deng_population(laplace_kern)

zoo = {
    "ot": ot(),

    "deng_theoretical_row": deng_theoretical,

    "deng_official": deng_official(
        temperature=SIGMA,
        self_mask=True,
        self_mask_distance=1e6,
    ),

    "sinkhorn_proxy": sinkhorn_proxy(proxy_kern),

    "ms_unnorm_nm": norm_matched(
        mean_shift_unnormalized(laplace_kern),
        deng_theoretical,
    ),

    "sinkhorn_K0_row": sinkhorn_drift(SIGMA, K=0),
    "sinkhorn_K5": sinkhorn_drift(SIGMA, K=5),
    "sinkhorn_Kinf": sinkhorn_drift(
    SIGMA,
    K=np.inf,
    tol=1e-8,
    max_iter=4000,
    ),
}

if laplace_kern.has_sharp:
    zoo["sharp_core"] = sharp_core(laplace_kern)

zoo_runs = Runs()
for name, fld in zoo.items():
    probes = standard_probes(
        data, field_kernel(fld), with_certificates=ZOO_WITH_CERTIFICATES,
        eval_sigma=EVAL_SIGMA, r_cover=R_COVER, tau=TAU)
    zoo_runs[name] = run(
        data.q0, data.p, fld, n_iter=N_ITER, tau=TAU, clip=CLIP,
        probes=probes, probe_every=PROBE_EVERY, store_every=STORE_EVERY, label=name)

zoo_keys = ["W2", "Vnorm", "F", "n_covered", "H"]
if ZOO_WITH_CERTIFICATES:
    zoo_keys += ["ot_alignment"]
fig = plot_runs(zoo_runs, keys=zoo_keys, ncol=3, figsize=(12, 6))
save(fig, "01_zoo_curves");

In [ ]:
zoo_columns = ["W2", "Vnorm", "F", "H", "n_covered", "entropy", "precision"]
if ZOO_WITH_CERTIFICATES:
    zoo_columns += ["ot_alignment", "beta_star", "L_hat"]
table = runs_table(zoo_runs, columns=zoo_columns,
                   n_modes=len(data.centers), tau=TAU)
save_table(
    table, "01_drift_zoo",
    caption=(f"Drift zoo on {DATASET}: n_iter={N_ITER}, tau={TAU}."))
style(table)

In [ ]:
zoo_names = list(zoo_runs)
for page, start in enumerate(range(0, len(zoo_names), ZOO_PAGE_SIZE), 1):
    names = zoo_names[start:start + ZOO_PAGE_SIZE]
    page_runs = Runs({name: zoo_runs[name] for name in names})
    fig = plot_snapshots(page_runs, data.p,
                         ks=(0, N_ITER // 6, N_ITER // 2, N_ITER),
                         centers=data.centers, extent=data.extent,
                         r_cover=R_COVER, panel_size=SNAPSHOT_PANEL_SIZE)
    save(fig, f"01_zoo_snapshots_{page}")
    display(fig)
    plt.close(fig)

---
## 4 · Bandwidth annealing


In [ ]:
sigma_runs = Runs()
for s in BANDWIDTHS:
    fld = make_drift(BANDWIDTH_DRIFT, BANDWIDTH_KERNEL, s,
                     **BANDWIDTH_DRIFT_KWARGS)
    sigma_runs[f"sigma_{s}"] = run(
        data.q0, data.p, fld, n_iter=N_ITER, tau=TAU, clip=CLIP,
        probes=standard_probes(data, field_kernel(fld), eval_sigma=EVAL_SIGMA,
                               r_cover=R_COVER),
        probe_every=PROBE_EVERY, store_every=20, label=rf"$\sigma={s}$")

sigma_runs["anneal"] = run(
    data.q0, data.p,
    lambda s: make_drift(BANDWIDTH_DRIFT, BANDWIDTH_KERNEL, s,
                         **BANDWIDTH_DRIFT_KWARGS),
    schedule=exp_anneal(ANNEAL_START, ANNEAL_END, ANNEAL_KAPPA),
    n_iter=N_ITER, tau=TAU, clip=CLIP,
    probes=standard_probes(data, eval_sigma=EVAL_SIGMA, r_cover=R_COVER),
    probe_every=PROBE_EVERY, store_every=20,
    label=rf"exp anneal ${ANNEAL_START:g}\to{ANNEAL_END:g}$")

fig = plot_runs(sigma_runs, keys=("W2", "H", "n_covered", "entropy"))
save(fig, "01_bandwidth")
runs_table(sigma_runs, columns=("W2", "H", "n_covered", "entropy", "precision"),
           n_modes=len(data.centers))